# Dimension 3.3 - Resource Allocation Optimization
Max output, Rawlsian maximin, multi-objective Pareto optimization.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hdi.models.optimization import (
    optimize_allocation_max_output,
    optimize_allocation_maximin
)
from hdi.data.loaders import load_health_inputs, load_health_outcomes

## 1. Model 1: Maximize Aggregate Health Output

In [ ]:
# Load current allocations
inputs_df = load_health_inputs()
outcomes_df = load_health_outcomes()

# Maximize total health output (weighted sum of LE gains + U5MR reductions)
# subject to total budget constraint = current global total
result_max = optimize_allocation_max_output(
    current_inputs=inputs_df,
    current_outcomes=outcomes_df,
    budget_constraint=inputs_df["health_exp_per_capita"].sum()
)

# Compare current vs optimal allocation
comparison = pd.DataFrame({
    "current": inputs_df["health_exp_per_capita"],
    "optimal": result_max["optimal_allocation"],
    "change_pct": (result_max["optimal_allocation"] / inputs_df["health_exp_per_capita"] - 1) * 100
})

print(f"Current total output index: {result_max['current_objective']:.2f}")
print(f"Optimal total output index: {result_max['optimal_objective']:.2f}")
print(f"Improvement: {result_max['improvement_pct']:.1f}%")
print()
print("Top 10 countries with largest reallocation (% change):")
comparison.sort_values("change_pct", ascending=False).head(10)

## 2. Model 2: Rawlsian Maximin

In [ ]:
# Rawlsian maximin: maximize the minimum health outcome across all countries
# This prioritizes the worst-off country
result_maximin = optimize_allocation_maximin(
    current_inputs=inputs_df,
    current_outcomes=outcomes_df,
    budget_constraint=inputs_df["health_exp_per_capita"].sum()
)

comparison_maximin = pd.DataFrame({
    "current": inputs_df["health_exp_per_capita"],
    "optimal_maximin": result_maximin["optimal_allocation"],
    "change_pct": (result_maximin["optimal_allocation"] / inputs_df["health_exp_per_capita"] - 1) * 100
})

print(f"Current minimum health index: {result_maximin['current_min']:.4f}")
print(f"Optimal minimum health index: {result_maximin['optimal_min']:.4f}")
print(f"Improvement in worst-off: {result_maximin['improvement_pct']:.1f}%")
print()
print("Largest gainers under maximin:")
comparison_maximin.sort_values("change_pct", ascending=False).head(10)

## 3. Model 3: Multi-Objective Pareto (NSGA-II)

In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize as pymoo_minimize
from pymoo.termination import get_termination


class HealthAllocationProblem(Problem):
    """Multi-objective: maximize efficiency (neg) and maximize equity (neg)."""

    def __init__(self, current_inputs, current_outcomes, total_budget):
        self.current_inputs = current_inputs
        self.current_outcomes = current_outcomes
        self.total_budget = total_budget
        n_countries = len(current_inputs)
        super().__init__(
            n_var=n_countries,
            n_obj=2,  # efficiency, equity
            n_constr=1,  # budget constraint
            xl=np.zeros(n_countries),
            xu=np.full(n_countries, total_budget * 0.5)
        )

    def _evaluate(self, X, out, *args, **kwargs):
        # Objective 1: negative total health output (minimize = maximize output)
        f1 = -np.sum(X * self.current_outcomes.values, axis=1)
        # Objective 2: Gini coefficient of outcomes (minimize = more equity)
        f2 = np.array([self._gini(x) for x in X])
        # Budget constraint: sum of allocations <= total_budget
        g1 = np.sum(X, axis=1) - self.total_budget
        out["F"] = np.column_stack([f1, f2])
        out["G"] = g1.reshape(-1, 1)

    @staticmethod
    def _gini(x):
        x_sorted = np.sort(x)
        n = len(x_sorted)
        index = np.arange(1, n + 1)
        return (2 * np.sum(index * x_sorted) - (n + 1) * np.sum(x_sorted)) / (n * np.sum(x_sorted) + 1e-12)


# Setup and run NSGA-II
total_budget = inputs_df["health_exp_per_capita"].sum()
problem = HealthAllocationProblem(inputs_df, outcomes_df["life_expectancy"], total_budget)

algorithm = NSGA2(pop_size=100)
termination = get_termination("n_gen", 200)

res = pymoo_minimize(problem, algorithm, termination, seed=42, verbose=False)

print(f"Pareto front size: {len(res.F)}")
plt.figure(figsize=(8, 5))
plt.scatter(-res.F[:, 0], res.F[:, 1], c="steelblue", edgecolors="k", alpha=0.7)
plt.xlabel("Total Health Output (higher = better)")
plt.ylabel("Gini Coefficient (lower = more equitable)")
plt.title("Pareto Front: Efficiency vs Equity")
plt.tight_layout()
plt.show()

## 4. Reallocation Sankey

In [ ]:
import plotly.graph_objects as go

# Show reallocation flows from current to optimal (max output model)
# Identify top donors (reduced allocation) and top recipients (increased allocation)
changes = comparison["change_pct"].dropna()
donors = changes[changes < -5].sort_values().head(10)  # top 10 countries losing funds
recipients = changes[changes > 5].sort_values(ascending=False).head(10)  # top 10 gaining

# Build Sankey diagram
labels = list(donors.index) + ["Reallocation Pool"] + list(recipients.index)
n_donors = len(donors)
n_recipients = len(recipients)
pool_idx = n_donors  # index of the pool node

sources = list(range(n_donors)) + [pool_idx] * n_recipients
targets = [pool_idx] * n_donors + list(range(pool_idx + 1, pool_idx + 1 + n_recipients))
values = list((-donors).values) + list(recipients.values)

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=15, thickness=20),
    link=dict(source=sources, target=targets, value=values)
))
fig.update_layout(title="Reallocation Flows: Current to Optimal (Max Output)", font_size=12)
fig.show()

## 5. Sensitivity: Budget +/-20%

In [ ]:
# Run optimization at 0.8x, 1.0x, and 1.2x of current total budget
budget_multipliers = [0.8, 1.0, 1.2]
sensitivity_results = []

for mult in budget_multipliers:
    budget = total_budget * mult
    res_sens = optimize_allocation_max_output(
        current_inputs=inputs_df,
        current_outcomes=outcomes_df,
        budget_constraint=budget
    )
    sensitivity_results.append({
        "budget_multiplier": mult,
        "budget_total": budget,
        "optimal_objective": res_sens["optimal_objective"],
        "improvement_pct": res_sens["improvement_pct"]
    })

sens_df = pd.DataFrame(sensitivity_results)
print("Sensitivity Analysis: Budget +/-20%")
print(sens_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(sens_df["budget_multiplier"].astype(str) + "x", sens_df["optimal_objective"], color=["#e74c3c", "#3498db", "#2ecc71"])
ax.set_xlabel("Budget Multiplier")
ax.set_ylabel("Optimal Health Output Index")
ax.set_title("Sensitivity of Optimal Output to Budget Level")
plt.tight_layout()
plt.show()